In [1]:
print("ok")

ok


In [1]:
import os

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_pinecone import PineconeVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [8]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
def load_pdf(data):
    loader = PyPDFDirectoryLoader(data)
    documents = loader.load()
    return documents

In [3]:
extracted_data = load_pdf("../data")

In [ ]:
# extracted_data

In [4]:
def split_text(data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks = text_splitter.split_documents(data)
    return text_chunks

In [5]:
text_chunks = split_text(extracted_data)
print(len(text_chunks))

5860


In [6]:
def download_embedding_model():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    model_kwargs = {"device": "cpu"}

    embeddings = HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs)

    return embeddings

In [9]:
embeddings = download_embedding_model()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={'device': 'cpu'}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [11]:
query_result = embeddings.embed_query("Hello world")
print(len(query_result))

384


In [12]:
query_result[:3]

[-0.03447720408439636, 0.031023239716887474, 0.00673496862873435]

In [13]:
KEY = os.getenv("PINECONE_API_KEY")
index_name = os.getenv("PINECONE_INDEX_NAME")

In [ ]:
docsearch = PineconeVectorStore.from_texts(
    texts=[t.page_content for t in text_chunks],
    embedding=embeddings,
    index_name=index_name,
)

In [15]:
docsearch = PineconeVectorStore(index_name=index_name, embedding=embeddings)

In [34]:
query = "What are allergies?"
docs = docsearch.similarity_search(query, k=3)

In [35]:
for doc in docs:
    print("---")
    print(doc.page_content)


---
reaction. Allergic rhinitis is characterized by an itchy,
runny nose, often with a scratchy or irritated throat due
to post-nasal drip. Inflammation of the thin membrane
covering the eye (allergic conjunctivitis) causes redness,
irritation, and increased tearing in the eyes. Asthma caus-
es wheezing, coughing, and shortness of breath. Symp-
toms of food allergies depend on the tissues most sensi-
tive to the allergen and whether the allergen spread sys-
---
reactions is triggered by harmless, everyday substances.
This is the condition known as allergy, and the offend-
ing substance is called an allergen. Common inhaled
allergens include pollen, dust, and insect parts from tiny
house mites. Common food allergens include nuts, fish,
and milk.
Allergic reactions involve a special set of cells in
the immune system known as mast cells. Mast cells
serve as guards in the tissues where the body meets the
---
Purpose
Allergy is a reaction of the immune system. Nor-
mally, the immune system 

In [16]:
prompt_template = """
Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context: {context}
Question: {question}

Only return the helpful answer below and nothing else.
Helpful answer:
"""

In [17]:
prompt = ChatPromptTemplate.from_template(prompt_template)

In [18]:
llm = ChatOllama(
    model="llama3.2:3b",
    temperature=0.8,
    num_predict=512,
    streaming=True,
)

In [19]:
retriever = docsearch.as_retriever(search_kwargs={"k": 3})

In [20]:
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


In [21]:
for chunk in chain.stream("What is acne?"):
    print(chunk, end="", flush=True)

Acne is a common skin disease characterized by pimples on the face, chest, and back. It occurs when the pores of the skin become clogged with oil, dead skin cells, and bacteria.